# Kipp-Bericht: fester SMA20/SMA50-Test
Zellen nacheinander starten. Keine echten Ergebnisse vorweggenommen.
30 festgelegte Aktien; 2020–2024; Close ohne automatische Dividendenanpassung.
Bei Fehlern keine Parameter ändern, sondern den Status dokumentieren.

In [ ]:
%pip -q install yfinance pandas numpy

In [ ]:
from pathlib import Path
source = '#!/usr/bin/env python3\n"""Decision-Delta: fixed SMA20/SMA50 daily-close experiment."""\nimport argparse, csv, hashlib, html, json, platform, sys, time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\n\nTICKERS = [\'AAPL\',\'MSFT\',\'NVDA\',\'AMZN\',\'GOOGL\',\'META\',\'TSLA\',\'JPM\',\'XOM\',\'KO\',\n\'0005.HK\',\'0700.HK\',\'1398.HK\',\'0939.HK\',\'0941.HK\',\'1299.HK\',\'2318.HK\',\'0388.HK\',\'0883.HK\',\'3988.HK\',\n\'HSBA.L\',\'BP.L\',\'SHEL.L\',\'AZN.L\',\'ULVR.L\',\'GSK.L\',\'RIO.L\',\'BARC.L\',\'LLOY.L\',\'VOD.L\']\nSTART, END_EXCLUSIVE = \'2020-01-01\', \'2025-01-01\'\nBUY, SELL, NONE, INVALID = \'KAUF\', \'VERKAUF\', \'KEINE AKTION\', \'NICHT BERECHENBAR\'\nFIELDS = [\'Open\',\'High\',\'Low\',\'Close\',\'Adj Close\',\'Volume\',\'Dividends\',\'Stock Splits\',\'Capital Gains\']\nWARNING = (\'Vergleich repair=False gegen repair=True, KEIN Nachweis einer nachtraeglichen \'\n\'Anbieter-Korrektur oder eines echten ausgefuehrten Trades. Reparaturen koennen falsch sein. \'\n\'Getrennte Abrufe koennen zusaetzlich unterschiedliche Anbieterantworten enthalten. \'\n\'Close wird mit auto_adjust=False benutzt; Adj Close ist kein Strategieeingang. \'\n\'Yahoo Close ist trotzdem nicht garantiert historisch unbereinigt (z.B. Splits).\')\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\ndef write_csv(df, path):\n    df.to_csv(path, index=False, encoding=\'utf-8-sig\', float_format=\'%.17g\')\n\ndef decision(df):\n    c = pd.to_numeric(df[\'Close\'], errors=\'coerce\')\n    a = c.rolling(20, min_periods=20).mean()\n    b = c.rolling(50, min_periods=50).mean()\n    d = a-b\n    valid = d.notna() & d.shift().notna()\n    out = pd.Series(INVALID, index=df.index, dtype=object)\n    out.loc[valid] = NONE\n    out.loc[valid & (d>0) & (d.shift()<=0)] = BUY\n    out.loc[valid & (d<0) & (d.shift()>=0)] = SELL\n    return pd.DataFrame({\'SMA20\':a,\'SMA50\':b,\'Entscheidung\':out})\n\ndef equal(a,b):\n    if pd.isna(a) and pd.isna(b): return True\n    if pd.isna(a) or pd.isna(b): return False\n    return bool(a == b)\n\ndef field_diffs(a,b,ticker=\'TEST\'):\n    rows=[]\n    for dt in a.index.union(b.index).sort_values():\n        if dt not in a.index or dt not in b.index:\n            rows.append(dict(Aktie=ticker, Datum=str(dt.date()), Feld=\'ZEILE_FEHLT\',\n                Original=\'vorhanden\' if dt in a.index else \'fehlt\',\n                Repariert=\'vorhanden\' if dt in b.index else \'fehlt\'))\n            continue\n        for field in FIELDS:\n            if field not in a and field not in b: continue\n            av = a.at[dt,field] if field in a else np.nan\n            bv = b.at[dt,field] if field in b else np.nan\n            if not equal(av,bv):\n                rows.append(dict(Aktie=ticker,Datum=str(dt.date()),Feld=field,Original=av,Repariert=bv))\n    return rows\n\ndef validate(df):\n    if not isinstance(df.index,pd.DatetimeIndex): raise ValueError(\'Kein Datumsindex\')\n    if df.index.has_duplicates: raise ValueError(\'Doppelte Handelstage\')\n    if not df.index.is_monotonic_increasing: raise ValueError(\'Nicht chronologisch\')\n    if \'Close\' not in df: raise ValueError(\'Close fehlt\')\n    x=pd.to_numeric(df.Close,errors=\'coerce\')\n    if x.isna().any() or not np.isfinite(x).all() or (x<=0).any():\n        raise ValueError(\'Close fehlt, ist unendlich oder nicht positiv; keine Auffuellung erlaubt\')\n\ndef cause(a,b,dt):\n    """Individual reversion, oldest date first. Includes previous 50 sessions AND today."""\n    i=a.index.get_loc(dt)\n    expected=decision(a).at[dt,\'Entscheidung\']\n    candidates=[]\n    for j in range(max(0,i-50),i+1):\n        when=a.index[j]\n        av,bv=a.at[when,\'Close\'],b.at[when,\'Close\']\n        if not equal(av,bv): candidates.append((when,av,bv))\n    if not candidates:\n        return dict(Kategorie=\'keine Datenursache\',Ursache_Datum=\'\',Feld=\'\',Alter_Wert=\'\',Neuer_Wert=\'\',\n                    Einzel_Ruecktausch_Treffer=0,Weitere_Treffer=\'\',Kandidaten=0)\n    hits=[]\n    for when,av,bv in candidates:\n        trial=b.copy(deep=True)\n        trial.at[when,\'Close\']=av\n        if decision(trial).at[dt,\'Entscheidung\']==expected: hits.append((when,av,bv))\n    if not hits:\n        return dict(Kategorie=\'mehrere Änderungen wirken zusammen\',Ursache_Datum=\'\',Feld=\'\',\n                    Alter_Wert=\'\',Neuer_Wert=\'\',Einzel_Ruecktausch_Treffer=0,Weitere_Treffer=\'\',\n                    Kandidaten=len(candidates))\n    when,av,bv=hits[0]\n    # A reverse-direction test distinguishes a restoring value from a sufficient cause.\n    forward=a.copy(deep=True); forward.at[when,\'Close\']=bv\n    sufficient=decision(forward).at[dt,\'Entscheidung\']==decision(b).at[dt,\'Entscheidung\']\n    return dict(Kategorie=\'einzelner Rücktausch stellt Original wieder her\',\n        Ursache_Datum=str(when.date()),Feld=\'Close\',Alter_Wert=float(av),Neuer_Wert=float(bv),\n        Einzel_Ruecktausch_Treffer=len(hits),\n        Weitere_Treffer=\'; \'.join(str(x[0].date()) for x in hits[1:]),\n        Kandidaten=len(candidates),Auch_vorwaerts_ausreichend=bool(sufficient))\n\ndef position_effects(da,db):\n    """Separate illustrative long/flat state; does NOT change fixed signal strategy."""\n    pa=pb=0; rows=[]\n    for dt in da.index:\n        x,y=da.at[dt,\'Entscheidung\'],db.at[dt,\'Entscheidung\']\n        if x==INVALID or y==INVALID: continue\n        if x==BUY: pa=1\n        elif x==SELL: pa=0\n        if y==BUY: pb=1\n        elif y==SELL: pb=0\n        if pa!=pb:\n            rows.append(dict(Datum=str(dt.date()),Original_Position=pa,Repariert_Position=pb,\n                Typ=\'reine Positionsfolgewirkung\' if x==y else \'direkter Signalunterschied\'))\n    return rows\n\ndef analyse(a,b,ticker):\n    validate(a);validate(b)\n    diffs=field_diffs(a,b,ticker)\n    da,db=decision(a),decision(b)\n    if not a.index.equals(b.index):\n        return dict(status=\'BLOCKIERT: unterschiedliche Handelstage; keine Kausalzuordnung\',\n                    diffs=diffs,flips=[],positions=[],original=da,repaired=db,comparable=0)\n    rows=[]; first=\'\'\n    for dt in a.index:\n        x,y=da.at[dt,\'Entscheidung\'],db.at[dt,\'Entscheidung\']\n        if x==INVALID or y==INVALID or x==y: continue\n        if not first: first=str(dt.date())\n        row=dict(Aktie=ticker,Datum=str(dt.date()),Original=x,Repariert=y,\n                 Einordnung=\'erste direkte Kipp-Entscheidung\' if str(dt.date())==first else \'weiterer direkter Signalunterschied\',\n                 Erste_Kippentscheidung=first)\n        row.update(cause(a,b,dt));rows.append(row)\n    return dict(status=\'OK\',diffs=diffs,flips=rows,positions=position_effects(da,db),\n                original=da,repaired=db,\n                comparable=int(((da.Entscheidung!=INVALID)&(db.Entscheidung!=INVALID)).sum()))\n\ndef artificial():\n    dates=pd.bdate_range(\'2020-01-01\',periods=71)\n    base=pd.DataFrame({\'Close\':100.,\'Volume\':1000.},index=dates)\n    return base\n\ndef tests(out):\n    out.mkdir(parents=True,exist_ok=True)\n    records=[]\n    def check(name,expected,fn):\n        try:\n            actual,ok=fn()\n        except Exception as e: actual,ok=repr(e),False\n        records.append(dict(Test=name,Erwartet=expected,Tatsaechlich=str(actual),Richtig=\'JA\' if ok else \'NEIN\'))\n    def nflip(a,b): return len(analyse(a,b,\'SYNTHETISCH\')[\'flips\'])\n    a=artificial();b=a.copy()\n    check(\'01 Identische Daten\',\'0 Kipps\',lambda:(nflip(a,b),nflip(a,b)==0))\n    a=artificial();b=a.copy();b.iloc[-1,b.columns.get_loc(\'Volume\')]=999999\n    check(\'02 Nur Volumen geaendert\',\'1 Datenunterschied; 0 Kipps\',\n        lambda:((len(field_diffs(a,b)),nflip(a,b)),len(field_diffs(a,b))==1 and nflip(a,b)==0))\n    a=artificial();b=a.copy();b.iloc[-1,0]=110\n    def single():\n        r=analyse(a,b,\'SYNTHETISCH\')[\'flips\']; ok=len(r)==1 and r[0][\'Ursache_Datum\']==str(a.index[-1].date()) and r[0][\'Feld\']==\'Close\' and r[0][\'Original\']==NONE and r[0][\'Repariert\']==BUY\n        return [(x[\'Datum\'],x[\'Original\'],x[\'Repariert\'],x[\'Ursache_Datum\']) for x in r],ok\n    check(\'03 Ein Close verschiebt Kreuzung\',\'genau 1 Kipp, letzte Kerze als Ursache\',single)\n    a=artificial();a.iloc[-2,0]=99;a.iloc[-1,0]=103;b=a.copy();b.iloc[-26,0]=110;b.iloc[-27,0]=110\n    def jointly():\n        c=cause(a,b,a.index[-1]);x=decision(a).iloc[-1].Entscheidung;y=decision(b).iloc[-1].Entscheidung\n        return (x,y,c[\'Kategorie\']),x==BUY and y==NONE and c[\'Kategorie\']==\'mehrere Änderungen wirken zusammen\'\n    check(\'04 Zwei Ruecktausche erforderlich\',\'KAUF -> KEINE AKTION; mehrere Änderungen wirken zusammen\',jointly)\n    a=artificial();b=a.copy();b.iloc[-61,0]=200\n    def outside():\n        c=cause(a,b,a.index[-1]);return c[\'Kategorie\'],c[\'Kategorie\']==\'keine Datenursache\' and decision(a).iloc[-1].Entscheidung==decision(b).iloc[-1].Entscheidung\n    check(\'05 Aenderung 60 Handelstage vorher\',\'am Zieldatum keine Datenursache, gleiche Entscheidung\',outside)\n    a=artificial();b=a.copy();b.iloc[-1,0]=10000\n    check(\'06 100-facher Einzelpreis\',\'1 Kipp, letzter Close als Ursache\',\n        lambda:((nflip(a,b),cause(a,b,a.index[-1])[\'Ursache_Datum\']),nflip(a,b)==1 and cause(a,b,a.index[-1])[\'Ursache_Datum\']==str(a.index[-1].date())))\n    a=artificial();b=a.drop(a.index[-10])\n    def missing():\n        r=analyse(a,b,\'SYNTHETISCH\');return (r[\'status\'],len(r[\'flips\']),len(r[\'diffs\'])),r[\'status\'].startswith(\'BLOCKIERT\') and len(r[\'flips\'])==0 and any(x[\'Feld\']==\'ZEILE_FEHLT\' for x in r[\'diffs\'])\n    check(\'07 Fehlender Handelstag\',\'BLOCKIERT; ZEILE_FEHLT; keine scheinbaren Kipps\',missing)\n    a=artificial();a.iloc[-2,0]=101;b=a.copy();b.iloc[-51,0]=110\n    def boundary():\n        c=cause(a,b,a.index[-1]);x=decision(a).iloc[-1].Entscheidung;y=decision(b).iloc[-1].Entscheidung\n        return (x,y,c[\'Ursache_Datum\']),x==NONE and y==BUY and c[\'Ursache_Datum\']==str(a.index[-51].date())\n    check(\'08 Rand t-50 beeinflusst gestrigen SMA50\',\'KEINE AKTION -> KAUF; Ursache genau t-50\',boundary)\n    a=artificial();b=a.copy();b.iloc[60,0]=110\n    def carry():\n        r=analyse(a,b,\'SYNTHETISCH\');dt=str(a.index[61].date())\n        p=[x for x in r[\'positions\'] if x[\'Datum\']==dt]\n        s=[x for x in r[\'flips\'] if x[\'Datum\']==dt]\n        return (p,len(s)),len(s)==0 and len(p)==1 and p[0][\'Typ\']==\'reine Positionsfolgewirkung\'\n    check(\'09 Position wirkt nach, Signal nicht\',\'Folgewirkung im Positionsprotokoll; kein Signal-Kipp an t+1\',carry)\n    a=artificial();b=a.copy();b.iloc[-1,0]=np.nan\n    def invalid():\n        try: analyse(a,b,\'SYNTHETISCH\'); return \'falsch akzeptiert\',False\n        except ValueError: return \'BLOCKIERT: fehlender Close\',True\n    check(\'10 Fehlender Close darf nicht aufgefuellt werden\',\'BLOCKIERT\',invalid)\n    df=pd.DataFrame(records);write_csv(df,out/\'tests.csv\')\n    (out/\'tests.html\').write_text(\'<meta charset="utf-8"><h1>10 kuenstliche Tests</h1><p>Keine Marktdaten.</p>\'+df.to_html(index=False,escape=True),encoding=\'utf-8\')\n    print(df[[\'Test\',\'Richtig\']].to_string(index=False))\n    if (df.Richtig!=\'JA\').any(): raise RuntimeError(\'Mindestens ein Test fehlgeschlagen: kein Marktdatenlauf erlaubt\')\n    return df\n\ndef normalise(df):\n    if df is None or df.empty: raise ValueError(\'Leere Anbieterantwort\')\n    df=df.copy()\n    if isinstance(df.columns,pd.MultiIndex):\n        if df.columns.nlevels==2: df.columns=df.columns.get_level_values(0)\n        else: raise ValueError(\'Unerwarteter MultiIndex\')\n    idx=pd.DatetimeIndex(df.index)\n    if idx.tz is not None: idx=idx.tz_localize(None)\n    df.index=idx.normalize();df.index.name=\'Date\'\n    return df\n\ndef run(root):\n    import yfinance as yf\n    stamp=datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S_%fZ\')\n    out=root/(\'lauf_\'+stamp);out.mkdir(parents=True,exist_ok=False)\n    tests(out/\'tests\')\n    manifest=dict(created_utc=stamp,tickers=TICKERS,start=START,end_inclusive=\'2024-12-31\',\n        end_exclusive=END_EXCLUSIVE,interval=\'1d\',auto_adjust=False,back_adjust=False,\n        actions=True,keepna=True,repair=[False,True],strategy=\'SMA20 crosses SMA50; Close; no positions\',\n        warmup=\'Keine Daten vor 2020; erste 50 Zeilen ohne Entscheidung\',\n        causal_window=\'50 vorherige Handelstage PLUS Zieldatum = 51 Schlusskurse\',\n        candidate_order=\'Datum aufsteigend; Feld Close\',float_comparison=\'exakt; CSV roundtrip 17 Stellen\',\n        python=sys.version,pandas=pd.__version__,numpy=np.__version__,yfinance=yf.__version__,\n        warning=WARNING,source_sha256=sha(__file__),downloads=[],scope=\'Keine Strategieaenderung; keine Ausfuehrung/PnL\')\n    (out/\'experiment_vorab.json\').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n    summaries=[];alld=[];allf=[];allp=[]\n    for ticker in TICKERS:\n        print(\'Lade\',ticker,flush=True);data={};errs=[]\n        for repaired in (False,True):\n            mode=\'repair_true\' if repaired else \'repair_false\'\n            begin=datetime.now(timezone.utc).isoformat()\n            record=dict(ticker=ticker,repair=repaired,started_utc=begin)\n            try:\n                df=yf.download(ticker,start=START,end=END_EXCLUSIVE,interval=\'1d\',\n                    auto_adjust=False,back_adjust=False,repair=repaired,actions=True,keepna=True,\n                    threads=False,progress=False,multi_level_index=False,timeout=30)\n                df=normalise(df)\n                fn=out/f\'{ticker}_{mode}_{stamp}.csv\'\n                df.to_csv(fn,index=True,encoding=\'utf-8-sig\',float_format=\'%.17g\',date_format=\'%Y-%m-%d\')\n                # Analysis uses the persisted artifact, not a hidden in-memory variant.\n                stored=pd.read_csv(fn,index_col=\'Date\',parse_dates=[\'Date\'],float_precision=\'round_trip\')\n                data[repaired]=stored\n                record.update(file=fn.name,sha256=sha(fn),rows=len(stored),status=\'OK\')\n            except Exception as e:\n                errs.append(f\'{mode}: {type(e).__name__}: {e}\')\n                record.update(status=\'FEHLER\',error=errs[-1])\n            record[\'finished_utc\']=datetime.now(timezone.utc).isoformat()\n            manifest[\'downloads\'].append(record)\n            (out/\'manifest.json\').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n            time.sleep(2)\n        if len(data)!=2:\n            summaries.append(dict(Aktie=ticker,Status=\'DOWNLOAD FEHLER\',Fehler=\' | \'.join(errs),Datenunterschiede=None,Kipps=None,Vergleichbare_Tage=0));continue\n        a,b=data[False],data[True]\n        # Persist all per-day decisions even if comparison later fails validation.\n        for flag,df in data.items():\n            if \'Close\' in df:\n                decisions=decision(df).reset_index()\n                write_csv(decisions,out/f\'{ticker}_entscheidungen_repair_{str(flag).lower()}.csv\')\n        differences=field_diffs(a,b,ticker)\n        write_csv(pd.DataFrame(differences,columns=[\'Aktie\',\'Datum\',\'Feld\',\'Original\',\'Repariert\']),out/f\'{ticker}_datenunterschiede.csv\')\n        alld.extend(differences)\n        try:\n            r=analyse(a,b,ticker);allf.extend(r[\'flips\'])\n            allp.extend([dict(Aktie=ticker,**x) for x in r[\'positions\']])\n            summaries.append(dict(Aktie=ticker,Status=r[\'status\'],Datenunterschiede=len(differences),\n                Kipps=len(r[\'flips\']) if r[\'status\']==\'OK\' else None,Vergleichbare_Tage=r[\'comparable\']))\n        except Exception as e:\n            summaries.append(dict(Aktie=ticker,Status=\'BLOCKIERT: \'+str(e),Datenunterschiede=len(differences),Kipps=None,Vergleichbare_Tage=0))\n    sd=pd.DataFrame(summaries)\n    fc=[\'Aktie\',\'Datum\',\'Original\',\'Repariert\',\'Einordnung\',\'Erste_Kippentscheidung\',\'Kategorie\',\n        \'Ursache_Datum\',\'Feld\',\'Alter_Wert\',\'Neuer_Wert\',\'Einzel_Ruecktausch_Treffer\',\'Weitere_Treffer\',\'Kandidaten\',\'Auch_vorwaerts_ausreichend\']\n    fd=pd.DataFrame(allf).reindex(columns=fc)\n    dd=pd.DataFrame(alld,columns=[\'Aktie\',\'Datum\',\'Feld\',\'Original\',\'Repariert\'])\n    pd_=pd.DataFrame(allp,columns=[\'Aktie\',\'Datum\',\'Original_Position\',\'Repariert_Position\',\'Typ\'])\n    for name,frame in [(\'zusammenfassung\',sd),(\'kipp_liste\',fd),(\'datenunterschiede\',dd),(\'positionsfolgen_NUR_ILLUSTRATION\',pd_)]:\n        write_csv(frame,out/(name+\'.csv\'))\n    good=sd[sd.Status==\'OK\']; bad=len(sd)-len(good)\n    nchanges=int((pd.to_numeric(sd.Datenunterschiede,errors=\'coerce\')>0).sum())\n    nflip=len(fd)\n    leaders=\'Keine Aktie mit Kipp\' if nflip==0 else \', \'.join(fd.groupby(\'Aktie\').size().loc[lambda s:s==s.max()].index)\n    text=f\'Vorgesehen: 30 Aktien. Vollstaendig vergleichbar: {len(good)}. Fehler/blockiert: {bad}. Aktien mit erfassten Datenunterschieden: {nchanges}. Direkte Entscheidungs-Kipps: {nflip}. Meiste Kipps: {leaders}.\'\n    status=\'UNVOLLSTAENDIG\' if bad else \'VOLLSTAENDIG\'\n    sections=[\'<meta charset="utf-8"><style>body{font-family:Arial;max-width:1400px;margin:24px}table{border-collapse:collapse}td,th{border:1px solid #aaa;padding:5px}h1{color:#17445a}</style>\',\n        \'<h1>Kipp-Bericht: \'+status+\'</h1>\',\'<p>\'+html.escape(text)+\'</p>\',\n        \'<h2>Grenzen</h2><p>\'+html.escape(WARNING)+\'</p>\',\n        \'<p>Beobachtete Signalunterschiede sind keine ausgefuehrten Trades. Kein Renditebeweis. \'\n        \'Keine Positionsfolgen in der Kipp-Zaeh lung. Der separate Long/Flat-Zustand ist nur eine Illustration. \'\n        \'Fehlende Tage oder ungueltige Schlusskurse blockieren die Kausalzuordnung fuer die ganze Aktie. \'\n        \'Kein Auffuellen, keine Datumsverschiebung. Keine Daten vor 2020: die ersten 50 Beobachtungen sind Aufwaermphase. \'\n        \'Alle unterschiedlichen Entscheidungen sind direkte Unterschiede dieser zustandslosen Signalregel, \'\n        \'nicht automatisch Folgen eines frueheren Trades. Die Kategorie mehrere Aenderungen ist ein Einzel-Ruecktausch-Befund, \'\n        \'kein Beweis einer kleinsten gemeinsamen Ursachenmenge. Ein wiederherstellender Wert ist nicht zwingend alleinige Ursache.</p>\',\n        \'<h2>Aktien</h2>\',sd.to_html(index=False,escape=True),\'<h2>Alle Kipp-Entscheidungen</h2>\',fd.to_html(index=False,escape=True),\n        \'<h2>10 kuenstliche Tests</h2>\',pd.read_csv(out/\'tests/tests.csv\').to_html(index=False,escape=True)]\n    (out/\'bericht.html\').write_text(\'\\n\'.join(sections),encoding=\'utf-8\')\n    (out/\'ERGEBNIS.txt\').write_text(text+\'\\n\\n\'+WARNING,encoding=\'utf-8\')\n    manifest[\'result\']=dict(complete=(bad==0),comparable_tickers=len(good),blocked_or_failed=bad,flips=nflip)\n    manifest[\'files\']={p.name:sha(p) for p in out.iterdir() if p.is_file() and p.name!=\'manifest.json\'}\n    (out/\'manifest.json\').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding=\'utf-8\')\n    print(text);print(\'Bericht:\',out/\'bericht.html\')\n    return out\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--tests-only\',action=\'store_true\');p.add_argument(\'--out\',default=\'ergebnisse\');args=p.parse_args()\n    root=Path(args.out)\n    if args.tests_only: tests(root/\'tests\')\n    else: run(root)\nif __name__==\'__main__\':main()\n'
Path("kipp_beweis.py").write_text(source, encoding="utf-8")


## Echter Datenlauf
Zuerst werden die zehn künstlichen Tests ausgeführt. Nur wenn sie bestehen, werden die 60 Abrufe gestartet. Netzwerkfehler werden ausgewiesen. Ein erneuter Start ist ein neuer Versuch mit neuem Zeitstempel.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "kipp_beweis.py"], check=True)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("kipp_ergebnisse", "zip", "ergebnisse")
files.download("kipp_ergebnisse.zip")

## Interpretation
Öffne nach dem Entpacken **bericht.html**. Unvollständige Läufe nicht als Gesamtbeweis verwenden. Auch null Kipps ist ein ehrliches Ergebnis. Reparaturen können falsch sein; keine nachträgliche Anbieterrevision und keine realen Trades nachgewiesen. Vollständige Rohdaten nicht ohne Rechteprüfung öffentlich weitergeben.